# Validate and quarantine test chips
Execute the following cell to move/delete invalid Sentinel-2 chips.

In [2]:
#!/usr/bin/env python3
"""Sanity-check Sentinel-2 test chips and quarantine invalid files."""
from __future__ import annotations

import argparse
import logging
import shutil
from datetime import datetime
from pathlib import Path
try:
    NOTEBOOK_ROOT = Path(__file__).resolve().parent
except NameError:
    NOTEBOOK_ROOT = Path.cwd()
from typing import List, Optional, Tuple

import numpy as np
import pandas as pd
import rasterio
from rasterio.errors import RasterioIOError
from rasterio.warp import transform_bounds

LOGGER = logging.getLogger("validate_test_tiles")
MINIMUM_SIZE = 0 #set to 0, because the dataset was given to us
NOTEBOOK_ARGS: List[str] = []  # populate this list with CLI-style args when running interactively


def validate_tile(path: Path) -> Tuple[bool, Optional[str]]:
    try:
        with rasterio.open(path) as dataset:
            if dataset.count < 3:
                return False, "expected at least 3 bands"
            if min(dataset.width, dataset.height) < MINIMUM_SIZE:
                return False, "tile is smaller than expected"
            if dataset.dtypes[0] not in {"int16", "float32", "uint16"}:
                return False, f"unexpected dtype {dataset.dtypes[0]}"
            if dataset.read(masked=True).mask.all():
                return False, "all pixels are masked"
    except RasterioIOError as exc:
        return False, str(exc)
    return True, None


def parse_date_from_name(name: str) -> Optional[str]:
    parts = Path(name).stem.split("_")
    if len(parts) < 3:
        return None
    day, month, year = parts[-3], parts[-2], parts[-1]
    if not (day.isdigit() and month.isdigit() and year.isdigit()):
        return None
    try:
        date = datetime(int(year), int(month), int(day))
    except ValueError:
        return None
    return date.strftime("%Y-%m-%d")


def estimate_tile_size_km(min_lon: float, min_lat: float, max_lon: float, max_lat: float) -> float:
    mean_lat = 0.5 * (min_lat + max_lat)
    width_km = (max_lon - min_lon) * 111.32 * max(0.1, abs(np.cos(np.deg2rad(mean_lat))))
    height_km = (max_lat - min_lat) * 110.57
    return float(max(width_km, height_km))


def build_manifest(
    raw_dir: Path,
    labels_dir: Path,
    output_path: Path,
    dw_label_dir: Path,
    dw_prob_dir: Path,
    snow_label_dir: Path,
) -> Path:
    rows = []
    for tif in sorted(raw_dir.glob("*.tif")):
        label_path = labels_dir / tif.name
        if not label_path.exists():
            LOGGER.warning("Missing GT label for %s; skipping in manifest", tif.name)
            continue
        dw_label_path = dw_label_dir / tif.name
        dw_prob_path = dw_prob_dir / tif.name
        snow_path = snow_label_dir / tif.name
        if not dw_label_path.exists():
            LOGGER.warning("Missing Dynamic World label for %s", tif.name)
        if not dw_prob_path.exists():
            LOGGER.warning("Missing Dynamic World prob for %s", tif.name)
        if not snow_path.exists():
            LOGGER.warning("Missing derived snow label for %s", tif.name)
        with rasterio.open(tif) as dataset:
            bounds = dataset.bounds
            crs = dataset.crs
        if crs and str(crs) != "EPSG:4326":
            min_lon, min_lat, max_lon, max_lat = transform_bounds(crs, "EPSG:4326", *bounds, densify_pts=21)
        else:
            min_lon, min_lat, max_lon, max_lat = bounds
        acquisition_date = parse_date_from_name(tif.name)
        row = {
            "tile_id": tif.stem,
            "region_id": "test",
            "region_name": "test_dataset",
            "acquisition_date": acquisition_date or "",
            "bbox_min_lon": min_lon,
            "bbox_min_lat": min_lat,
            "bbox_max_lon": max_lon,
            "bbox_max_lat": max_lat,
            "tile_size_km": estimate_tile_size_km(min_lon, min_lat, max_lon, max_lat),
            "sentinel_path": f"raw/{tif.name}",
            "dynamic_world_label_path": f"all_labels/{tif.name}" if dw_label_path.exists() else "",
            "dynamic_world_prob_path": f"all_labels_probabilities/{tif.name}" if dw_prob_path.exists() else "",
            "snow_mask_path": f"snow_labels/{tif.name}" if snow_path.exists() else "",
            "given_gt_snow_mask_path": f"given_gt_snow_labels/{tif.name}",
        }
        rows.append(row)
    manifest = pd.DataFrame(rows)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    manifest.to_csv(output_path, index=False)
    LOGGER.info("Wrote manifest with %d entries to %s", len(manifest), output_path)
    return output_path


def quarantine_tiles(raw_dir: Path, invalid_dir: Path, delete: bool) -> List[Path]:
    invalid_dir.mkdir(parents=True, exist_ok=True)
    quarantined: List[Path] = []
    for tif in sorted(raw_dir.glob("*.tif")):
        ok, reason = validate_tile(tif)
        if ok:
            LOGGER.debug("%s looks good", tif.name)
            continue
        LOGGER.warning("%s marked invalid: %s", tif.name, reason)
        if delete:
            tif.unlink(missing_ok=True)
            LOGGER.info("Deleted %s", tif.name)
        else:
            destination = invalid_dir / tif.name
            shutil.move(str(tif), destination)
            LOGGER.info("Moved %s -> %s", tif.name, destination)
            quarantined.append(destination)
    return quarantined


def parse_args(argv: Optional[List[str]] = None) -> argparse.Namespace:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument(
        "--raw-dir",
        type=Path,
        default=NOTEBOOK_ROOT / "test_dataset" / "raw",
        help="Directory containing downloaded Sentinel-2 chips",
    )
    parser.add_argument(
        "--dw-label-dir",
        type=Path,
        default=NOTEBOOK_ROOT / "test_dataset" / "all_labels",
        help="Directory containing Dynamic World labels",
    )
    parser.add_argument(
        "--dw-prob-dir",
        type=Path,
        default=NOTEBOOK_ROOT / "test_dataset" / "all_labels_probabilities",
        help="Directory containing Dynamic World probabilities",
    )
    parser.add_argument(
        "--snow-label-dir",
        type=Path,
        default=NOTEBOOK_ROOT / "test_dataset" / "snow_labels",
        help="Directory containing derived snow masks",
    )
    parser.add_argument(
        "--labels-dir",
        type=Path,
        default=NOTEBOOK_ROOT / "test_dataset" / "given_gt_snow_labels",
        help="Directory containing given ground-truth snow masks",
    )
    parser.add_argument(
        "--manifest-path",
        type=Path,
        default=NOTEBOOK_ROOT / "test_dataset" / "manifest.csv",
        help="Manifest CSV path",
    )
    parser.add_argument(
        "--invalid-dir",
        type=Path,
        default=NOTEBOOK_ROOT / "test_dataset" / "invalid",
        help="Quarantine folder for invalid chips",
    )
    parser.add_argument(
        "--delete",
        action="store_true",
        help="Delete invalid tiles instead of moving them",
    )
    parser.add_argument(
        "--log-level",
        default="INFO",
        choices=("DEBUG", "INFO", "WARNING", "ERROR"),
    )
    return parser.parse_args(argv)


def main() -> None:
    args = parse_args(NOTEBOOK_ARGS or None)
    logging.basicConfig(level=getattr(logging, args.log_level))
    quarantined = quarantine_tiles(args.raw_dir, args.invalid_dir, args.delete)
    LOGGER.info("Quarantined %d files", len(quarantined))
    build_manifest(
        args.raw_dir,
        args.labels_dir,
        args.manifest_path,
        args.dw_label_dir,
        args.dw_prob_dir,
        args.snow_label_dir,
    )


if __name__ == "__main__":
    import sys
    if any(arg.startswith("-f") for arg in sys.argv[1:]):
        sys.argv = [sys.argv[0]]
    main()


INFO:validate_test_tiles:Quarantined 0 files
INFO:validate_test_tiles:Wrote manifest with 8 entries to d:\analiza_snezisc2\2_download_test_data_and_create_test_dataset\test_dataset\manifest.csv
